# DAU fell after an SDK release. Did player activity change, or did measurement break?

> **Fictional portfolio demonstration.** All data and incident details are synthetic. The results describe this simulation only, not a real game or business outcome.

## Decision
Diagnose an apparent drop in daily active users (DAU) after a staged analytics SDK release. Separate changes in true simulated activity from failures in event identity and delivery.

## Simulation design
The generator creates a stable player population with ordinary daily variation. After rollout, a subset of upgraded players has missing IDs on all their client events. It also injects retry duplicates and longer event delays. A separate truth table records the known active-user count by date and platform. This is an evaluation aid available only because the data are simulated; a real analyst would not have this oracle.

The rollout is staggered: Android starts on 12 May 2026 and iOS on 15 May 2026. Use event timestamps for activity date. The DAU proxy counts distinct non-null users with `app_open` or `session_start`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path.cwd()
if not (BASE / 'data').exists() and (BASE / '8-mobile-sdk-metric-drop' / 'data').exists():
    BASE = BASE / '8-mobile-sdk-metric-drop'
events = pd.read_csv(BASE / 'data/events.csv', parse_dates=['event_timestamp', 'received_timestamp'])
truth = pd.read_csv(BASE / 'data/simulation_truth.csv', parse_dates=['date'])
print(f'{len(events):,} event rows, {events.event_timestamp.min().date()} to {events.event_timestamp.max().date()}')
events.head()

## 1. Validate the data and define the metric
`user_id` blanks are expected in this simulation. Event IDs must remain unique, including retry rows. The dashboard-style DAU uses valid user IDs and qualifying activity events.

In [ ]:
required = {'event_id','user_id','event_name','event_timestamp','received_timestamp','platform','app_version','country','acquisition_channel'}
assert required <= set(events.columns)
assert events.event_id.is_unique
assert (events.received_timestamp >= events.event_timestamp).all()
events['event_date'] = events.event_timestamp.dt.floor('D')
events['delay_hours'] = (events.received_timestamp - events.event_timestamp).dt.total_seconds() / 3600
events['is_retry'] = events.event_id.str.endswith('_retry')
activity = events[events.event_name.isin(['app_open','session_start'])]
dau = (activity.dropna(subset=['user_id']).groupby(['event_date','platform']).user_id.nunique().rename('observed_dau').reset_index())
dau.head()

## 2. Compare observed DAU with the simulation truth
The truth table is hidden ground truth for this generated example. It lets us measure the impact of identity loss. In a real incident, use an independent source (for example, server-side activity) rather than assuming client data are complete.

In [ ]:
observed = dau.groupby('event_date').observed_dau.sum().rename('observed_dau').reset_index()
comparison = truth.merge(observed, left_on='date', right_on='event_date', how='left')
comparison['coverage'] = comparison.observed_dau / comparison.true_active_users
pre = comparison[comparison.date < '2026-05-12']
post = comparison[comparison.date >= '2026-05-15']
summary = pd.DataFrame({
    'period':['Before rollout','After both platforms rolled out'],
    'mean_true_dau':[pre.true_active_users.mean(),post.true_active_users.mean()],
    'mean_observed_dau':[pre.observed_dau.mean(),post.observed_dau.mean()],
    'mean_observed_coverage':[pre.coverage.mean(),post.coverage.mean()]
}).set_index('period').round(3)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(comparison.date, comparison.true_active_users, label='True simulated active users', color='black')
ax.plot(comparison.date, comparison.observed_dau, label='Observed client-event DAU', marker='.', linewidth=1)
ax.axvline(pd.Timestamp('2026-05-12'), color='darkorange', linestyle='--', label='Android rollout')
ax.axvline(pd.Timestamp('2026-05-15'), color='firebrick', linestyle=':', label='iOS rollout')
ax.set(title='Observed DAU falls below simulated activity after rollout', xlabel='Event date', ylabel='Distinct users')
ax.grid(alpha=.2); ax.legend(); plt.tight_layout()

## 3. Check event quality by date and app version
Missing IDs are simulated as a player-level problem after that player's platform rollout. Retry duplicates use a `_retry` suffix only because this generator controls the data. In production, use a documented idempotency key or a validated event fingerprint.

In [ ]:
quality = events.groupby(['event_date','platform','app_version']).agg(
    event_rows=('event_id','size'),
    missing_id_share=('user_id',lambda s:s.isna().mean()),
    retry_share=('is_retry','mean'),
    p95_delay_hours=('delay_hours',lambda s:s.quantile(.95))
).reset_index()
quality.head()

In [ ]:
daily = events.groupby('event_date').agg(
    missing_id_share=('user_id',lambda s:s.isna().mean()),
    retry_share=('is_retry','mean'),
    p95_delay_hours=('delay_hours',lambda s:s.quantile(.95))
)
fig, axes = plt.subplots(3,1,figsize=(12,8),sharex=True)
for col,ax,title in zip(daily.columns,axes,['Missing user ID share','Retry duplicate share of event rows','95th percentile delivery delay (hours)']):
    daily[col].plot(ax=ax,color='steelblue'); ax.set_ylabel(title); ax.grid(alpha=.2)
for ax in axes:
    ax.axvline(pd.Timestamp('2026-05-12'),color='darkorange',linestyle='--')
    ax.axvline(pd.Timestamp('2026-05-15'),color='firebrick',linestyle=':')
plt.tight_layout()

## 4. Locate the issue by platform and version
The staged rollout provides a useful diagnostic comparison, but it is not a randomized experiment. Differences by version are consistent with a measurement failure in this simulation; date alignment alone would not prove causation in a real product.

In [ ]:
version_check = events.groupby(['platform','app_version']).agg(
    rows=('event_id','size'),
    missing_id_share=('user_id',lambda s:s.isna().mean()),
    retry_share=('is_retry','mean'),
    p95_delay_hours=('delay_hours',lambda s:s.quantile(.95))
).round(3)
version_check

In [ ]:
platform_dau = dau.pivot(index='event_date',columns='platform',values='observed_dau')
platform_dau.plot(figsize=(12,4),marker='.',linewidth=1)
plt.axvline(pd.Timestamp('2026-05-12'),color='darkorange',linestyle='--',label='Android rollout')
plt.axvline(pd.Timestamp('2026-05-15'),color='firebrick',linestyle=':',label='iOS rollout')
plt.title('Observed DAU by platform'); plt.ylabel('Distinct non-null users'); plt.grid(alpha=.2); plt.legend(); plt.tight_layout()

## 5. Account for late-arriving events
Compare the final event-time metric with what would have been available 24 hours after each event date ended. This is a simple snapshot simulation, not a full warehouse freshness model. Events received after the cutoff can backfill historical DAU.

In [ ]:
activity = activity.copy()
activity['snapshot_cutoff'] = activity.event_date + pd.Timedelta(days=1)
available_24h = activity[(activity.received_timestamp < activity.snapshot_cutoff) & activity.user_id.notna()]
snapshot = available_24h.groupby('event_date').user_id.nunique().rename('available_after_24h')
latency_check = comparison.set_index('date')[['true_active_users','observed_dau']].join(snapshot)
latency_check['available_after_24h'] = latency_check.available_after_24h.fillna(0)
latency_check.tail(10)

## Interpretation

- **In this simulation:** true active users remain near the designed baseline, while observed DAU falls after rollout because some upgraded players lose their identifiers. Event retries inflate raw event rows but do not inflate distinct-user DAU in this metric. Delivery delays make recent counts incomplete at a 24-hour snapshot.
- **Business conclusion:** treat the DAU dashboard as degraded; do not respond by changing UA budgets or product features until the event pipeline is checked. Validate ID completeness, retry handling, delivery lag, and server/client agreement by app version.
- **Limits:** the rollout is not randomized; there is no real backend data, spend table, or causal impact estimate. The simulation truth is available only because we generated the data. Do not report these numbers as real findings.